In [8]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
koleksi_menu = db["menu"]

# Bersihkan data lama
koleksi_menu.delete_many({})

data_menu = [
    # 6 Menu Kopi (kepahitan 1-5)
    {"nama_menu": "Espresso", "harga": 15000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 20},
    {"nama_menu": "Americano", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 30},
    {"nama_menu": "Cappuccino", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 3},
    {"nama_menu": "Latte", "harga": 26000, "kategori": "Kopi", "tingkat_kepahitan": 2},
    {"nama_menu": "Kopi Tubruk", "harga": 12000, "kategori": "Kopi", "tingkat_kepahitan": 5},
    {"nama_menu": "Mocha", "harga": 28000, "kategori": "Kopi", "tingkat_kepahitan": 3},
    
    # 6 Menu Makanan (pedas_level 0-3)
    {"nama_menu": "Nasi Goreng Spesial", "harga": 30000, "kategori": "Makanan", "pedas_level": 2, "stok": 10},
    {"nama_menu": "Mie Tek-Tek", "harga": 20000, "kategori": "Makanan", "pedas_level": 3},
    {"nama_menu": "Kentang Goreng", "harga": 15000, "kategori": "Makanan", "pedas_level": 0},
    {"nama_menu": "Roti Bakar Cokelat", "harga": 18000, "kategori": "Makanan", "pedas_level": 0},
    {"nama_menu": "Ayam Geprek", "harga": 25000, "kategori": "Makanan", "pedas_level": 3},
    {"nama_menu": "Pisang Goreng Keju", "harga": 16000, "kategori": "Makanan", "pedas_level": 0}
]

# Insert 12 dokumen
koleksi_menu.insert_many(data_menu)
print(f"Berhasil menambahkan {koleksi_menu.count_documents({})} menu.")

Berhasil menambahkan 12 menu.


In [9]:
#Kopi dengan tingkat kepahitan di atas 3
print("Kopi sangat pahit (>3):")
for m in koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f"   - {m['nama_menu']} (Pahit: {m['tingkat_kepahitan']})")

#Harga di antara 15000 dan 30000
print("Harga antara 15.000 - 30.000:")
for m in koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
     print(f"   - {m['nama_menu']} (Rp{m['harga']:,})")

#Menu yang tidak memiliki field pedas level
print("Menu tanpa atribut pedas_level:")
for m in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f"   - {m['nama_menu']}")

Kopi sangat pahit (>3):
   - Espresso (Pahit: 5)
   - Americano (Pahit: 4)
   - Kopi Tubruk (Pahit: 5)
Harga antara 15.000 - 30.000:
   - Espresso (Rp15,000)
   - Americano (Rp18,000)
   - Cappuccino (Rp25,000)
   - Latte (Rp26,000)
   - Mocha (Rp28,000)
   - Nasi Goreng Spesial (Rp30,000)
   - Mie Tek-Tek (Rp20,000)
   - Kentang Goreng (Rp15,000)
   - Roti Bakar Cokelat (Rp18,000)
   - Ayam Geprek (Rp25,000)
   - Pisang Goreng Keju (Rp16,000)
Menu tanpa atribut pedas_level:
   - Espresso
   - Americano
   - Cappuccino
   - Latte
   - Kopi Tubruk
   - Mocha


In [10]:
#Tambahkan status "Tersedia" ke seluruh dokumen
koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})

#Ubah status menjadi "Habis" untuk 2 menu
koleksi_menu.update_one({"nama_menu": "Espresso"}, {"$set": {"status": "Habis"}})
koleksi_menu.update_one({"nama_menu": "Mie Tek-Tek"}, {"$set": {"status": "Habis"}})

UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

In [12]:
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql.functions import avg

spark = SparkSession.builder.appName("Tugas9-KopiNusantara").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Ambil data ke pandas DataFrame
dokumen_menu = list(koleksi_menu.find({}))
df_pandas = pd.DataFrame(dokumen_menu)
df_pandas["_id"] = df_pandas["_id"].astype(str)

# Konversi ke Spark DataFrame
df_spark = spark.createDataFrame(df_pandas)

print("Schema DataFrame:")
df_spark.printSchema()

print("Rata-rata Harga per Kategori:")
df_spark.groupBy("kategori").agg(avg("harga").alias("rata_rata_harga")).show()

Schema DataFrame:
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- stok: double (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)

Rata-rata Harga per Kategori:
+--------+------------------+
|kategori|   rata_rata_harga|
+--------+------------------+
|    Kopi|20666.666666666668|
| Makanan|20666.666666666668|
+--------+------------------+



Tantangan utama saat mengonversi data MongoDB ke Spark DataFrame adalah perbedaan standar struktur data. MongoDB menggunakan model NoSQL berkonsep flexible schema, sehingga dokumen dalam satu collection tidak wajib memiliki field yang seragam. Contohnya, menu "Kopi" memiliki tingkat_kepahitan, sedangkan "Makanan" memakai pedas_level. Sebaliknya, Spark DataFrame mewajibkan skema tabel yang kaku dan seragam di setiap barisnya. Saat dikonversi, atribut yang tidak ada pada suatu dokumen otomatis diisi dengan nilai kosong (null atau NaN). Untuk mengatasinya, saya melakukan pemrosesan awal menggunakan modul pandas agar tipe datanya disesuaikan terlebih dahulu. Setelah menjadi DataFrame Spark, penanganan dapat dilanjutkan dengan fungsi seperti .fillna() untuk mengganti nilai null, atau dibiarkan kosong asalkan tidak mengganggu fungsi agregasi seperti penghitungan rata-rata harga.